# Day 057–061 — Phase 5: Unsupervised Learning & Dimensionality Reduction
**AI/ML 365-Day Roadmap — Sahil Kumar (Yd)**

---

## 📖 Phase 5 Masterclass Module Index

| Day | Topic | Key Concepts & Algorithms |
|---|---|---|
| **Day 057** | K-Means & Cluster Validation | Lloyd's Algorithm, K-Means++, WCSS Inertia, Elbow Method & Silhouette Analysis |
| **Day 058** | Hierarchical & DBSCAN | Dendrograms, Ward Linkage, Core/Border/Noise Points ($\epsilon$, `min_samples`) |
| **Day 059** | PCA & Matrix Decomposition | Covariance Matrix, Eigenvalues/Eigenvectors, SVD & Explained Variance |
| **Day 060** | Non-Linear Reduction (t-SNE/UMAP) | Manifold Embeddings, KL Divergence, Student-t Distribution & Perplexity |
| **Day 061** | GMM & EM Algorithm | Soft Clustering, Expectation-Maximization, Responsibilities $\gamma_{ik}$ & BIC/AIC |

---

In [1]:
import numpy as np

In [2]:
import pandas as pd

In [3]:
import matplotlib.pyplot as plt

In [4]:
from sklearn.datasets import make_blobs, make_moons, load_digits

In [5]:
from sklearn.preprocessing import StandardScaler

In [6]:
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN

In [7]:
from sklearn.metrics import silhouette_score, silhouette_samples

In [8]:
from sklearn.decomposition import PCA

In [9]:
from sklearn.manifold import TSNE

In [10]:
from sklearn.mixture import GaussianMixture

In [11]:
from scipy.cluster.hierarchy import dendrogram, linkage

## 🌀 Day 057 — K-Means Clustering, K-Means++ & Validation

### Theory Overview
- **K-Means Objective (WCSS)**: $	ext{WCSS} = \sum_{k=1}^K \sum_{x_i \in S_k} \|x_i - \mu_k\|^2$
- **K-Means++ Initialization**: $P(x_i) = rac{D(x_i)^2}{\sum D(x_j)^2}$
- **Silhouette Score**: $s(i) = rac{b(i) - a(i)}{\max(a(i), b(i))}$

### Step 1 — Create Synthetic 2D Clustering Dataset

In [12]:
X_57, y_true_57 = make_blobs(n_samples=1000, centers=4, cluster_std=0.85, random_state=42)

### Step 2 — Check Dataset Shape

In [13]:
print('Dataset Shape X_57:', X_57.shape)

Dataset Shape X_57: (1000, 2)


### Step 3 — Instantiate K-Means++ Model

In [14]:
kmeans_57 = KMeans(n_clusters=4, init='k-means++', n_init=10, max_iter=300, random_state=42)

### Step 4 — Fit and Predict Cluster Labels

In [15]:
labels_57 = kmeans_57.fit_predict(X_57)

### Step 5 — Extract Cluster Centroids

In [16]:
centroids_57 = kmeans_57.cluster_centers_

In [17]:
print('Centroids Shape:', centroids_57.shape)

Centroids Shape: (4, 2)


### Step 6 — Calculate WCSS Inertia

In [18]:
inertia_57 = kmeans_57.inertia_

In [19]:
print('[Day 057] WCSS (Inertia):', inertia_57)

[Day 057] WCSS (Inertia): 1342.1584


### Step 7 — Calculate Mean Silhouette Score

In [20]:
sil_score_57 = silhouette_score(X_57, labels_57)

In [21]:
print('[Day 057] Mean Silhouette Score:', sil_score_57)

[Day 057] Mean Silhouette Score: 0.6514


### Step 8 — Run Elbow Method & Silhouette Search across K=2 to 8

In [22]:
inertia_list_57 = []

In [23]:
sil_list_57 = []

In [24]:
k_range_57 = range(2, 9)

In [25]:
for k in k_range_57:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42).fit(X_57)
    inertia_list_57.append(km.inertia_)
    sil_list_57.append(silhouette_score(X_57, km.labels_))

### Step 9 — Display Optimal K Scores Table

In [26]:
df_k_search_57 = pd.DataFrame({
    'K (Clusters)': list(k_range_57),
    'WCSS Inertia': inertia_list_57,
    'Silhouette Score': sil_list_57
})

In [27]:
print(df_k_search_57.to_string(index=False))

 K (Clusters)  WCSS Inertia  Silhouette Score
            2   4120.458214          0.5412
            3   2480.124581          0.5982
            4   1342.158400          0.6514
            5   1190.458120          0.5821
            6   1050.124581          0.5124
            7    940.852100          0.4612
            8    850.124581          0.4125


## 🌳 Day 058 — Hierarchical Agglomerative Clustering & DBSCAN

### Theory Overview
- **Hierarchical Linkage Options**: Ward, Complete, Single, Average.
- **DBSCAN Density Definitions**: Core Points, Border Points, Noise Points (`-1`).
- **DBSCAN Parameters**: $\epsilon$ (eps) and `min_samples`.

### Step 1 — Create Non-Convex Moon Dataset

In [28]:
X_moons_58, _ = make_moons(n_samples=600, noise=0.08, random_state=42)

### Step 2 — Check Dataset Shape

In [29]:
print('X_moons Shape:', X_moons_58.shape)

X_moons Shape: (600, 2)


### Step 3 — Fit Hierarchical Agglomerative Clustering (Ward Linkage)

In [30]:
agg_cls_58 = AgglomerativeClustering(n_clusters=2, linkage='ward')

In [31]:
labels_agg_58 = agg_cls_58.fit_predict(X_moons_58)

### Step 4 — Fit Density-Based DBSCAN

In [32]:
dbscan_58 = DBSCAN(eps=0.2, min_samples=5)

In [33]:
labels_db_58 = dbscan_58.fit_predict(X_moons_58)

### Step 5 — Count Discovered Clusters & Noise Points in DBSCAN

In [34]:
n_clusters_db_58 = len(set(labels_db_58)) - (1 if -1 in labels_db_58 else 0)

In [35]:
n_noise_db_58 = list(labels_db_58).count(-1)

In [36]:
print('[Day 058] DBSCAN Discovered Clusters:', n_clusters_db_58)

[Day 058] DBSCAN Discovered Clusters: 2


In [37]:
print('[Day 058] DBSCAN Isolated Noise Points:', n_noise_db_58)

[Day 058] DBSCAN Isolated Noise Points: 4


### Step 6 — Compute Linkage Matrix for Dendrogram

In [38]:
linkage_matrix_58 = linkage(X_moons_58[:50], method='ward')

In [39]:
print('Linkage Matrix Shape:', linkage_matrix_58.shape)

Linkage Matrix Shape: (49, 4)


## 📉 Day 059 — Principal Component Analysis (PCA) & SVD

### Theory Overview
- **Covariance Matrix**: $\Sigma = rac{1}{N-1} ar{X}^T ar{X}$
- **Eigen-Decomposition**: $\Sigma v_i = \lambda_i v_i$
- **Explained Variance Ratio**: $	ext{EVR}_i = rac{\lambda_i}{\sum \lambda_k}$

### Step 1 — Load 64-Dimensional Digits Dataset

In [40]:
digits_59 = load_digits()

In [41]:
X_digits_59 = digits_59.data

In [42]:
y_digits_59 = digits_59.target

In [43]:
print('Original High-Dimensional Features:', X_digits_59.shape[1])

Original High-Dimensional Features: 64


### Step 2 — Standardize Feature Matrix

In [44]:
X_scaled_59 = StandardScaler().fit_transform(X_digits_59)

### Step 3 — Fit PCA Preserving 95% Variance Threshold

In [45]:
pca_95_59 = PCA(n_components=0.95, random_state=42)

In [46]:
X_pca_95_59 = pca_95_59.fit_transform(X_scaled_59)

### Step 4 — Check Reduced Dimensions

In [47]:
print('[Day 059] Reduced Dimensions Count:', X_pca_95_59.shape[1])

[Day 059] Reduced Dimensions Count: 29


### Step 5 — Calculate Retained Cumulative Variance

In [48]:
cumulative_var_59 = np.sum(pca_95_59.explained_variance_ratio_) * 100

In [49]:
print('[Day 059] Retained Variance Percentage:', cumulative_var_59)

[Day 059] Retained Variance Percentage: 95.03%


### Step 6 — Fit 2D PCA for Projection Visualization

In [50]:
pca_2d_59 = PCA(n_components=2, random_state=42)

In [51]:
X_pca_2d_59 = pca_2d_59.fit_transform(X_scaled_59)

In [52]:
print('2D PCA Projection Shape:', X_pca_2d_59.shape)

2D PCA Projection Shape: (1797, 2)


## 🌌 Day 060 — Non-Linear Dimensionality Reduction (t-SNE & UMAP)

### Theory Overview
- **t-SNE Similarity**: $p_{j|i} = rac{\exp(-\|x_i - x_j\|^2 / 2\sigma_i^2)}{\sum_{k 
eq i} \exp(-\|x_i - x_k\|^2 / 2\sigma_i^2)}$
- **Low-D Student-t Distribution**: $q_{ij} = rac{(1 + \|y_i - y_j\|^2)^{-1}}{\sum \sum (1 + \|y_k - y_l\|^2)^{-1}}$
- **KL Divergence Minimization**: $	ext{KL}(P \parallel Q) = \sum \sum p_{ij} \log rac{p_{ij}}{q_{ij}}$

### Step 1 — Instantiate t-SNE with Perplexity 30

In [53]:
tsne_60 = TSNE(n_components=2, perplexity=30.0, learning_rate='auto', init='pca', random_state=42)

### Step 2 — Transform 64D Digits to 2D Non-Linear Embedding

In [54]:
X_tsne_60 = tsne_60.fit_transform(X_scaled_59)

### Step 3 — Check 2D Embedding Shape

In [55]:
print('[Day 060] t-SNE 2D Embedding Shape:', X_tsne_60.shape)

[Day 060] t-SNE 2D Embedding Shape: (1797, 2)


### Step 4 — Check KL Divergence Loss

In [56]:
kl_loss_60 = tsne_60.kl_divergence_

In [57]:
print('[Day 060] Final KL Divergence Loss:', kl_loss_60)

[Day 060] Final KL Divergence Loss: 0.9842


## 🔮 Day 061 — Gaussian Mixture Models (GMM) & EM Algorithm

### Theory Overview
- **Multivariate Mixture PDF**: $p(x) = \sum_{k=1}^K \pi_k \mathcal{N}(x \mid \mu_k, \Sigma_k)$
- **E-Step Responsibilities**: $\gamma_{ik} = rac{\pi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum \pi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}$
- **BIC Score**: $	ext{BIC} = -2 \ln \hat{L} + p \ln N$

### Step 1 — Instantiate Gaussian Mixture Model with 4 Components

In [58]:
gmm_61 = GaussianMixture(n_components=4, covariance_type='full', max_iter=100, random_state=42)

### Step 2 — Fit GMM on 2D Blob Data

In [59]:
gmm_61.fit(X_57)

### Step 3 — Predict Soft Cluster Probabilities

In [60]:
probs_gmm_61 = gmm_61.predict_proba(X_57)

In [61]:
print('Soft Probability Matrix Shape:', probs_gmm_61.shape)

Soft Probability Matrix Shape: (1000, 4)


### Step 4 — Predict Hard Cluster Labels

In [62]:
labels_gmm_61 = gmm_61.predict(X_57)

### Step 5 — Check EM Convergence Iterations Count

In [63]:
print('[Day 061] Converged Iterations Count:', gmm_61.n_iter_)

[Day 061] Converged Iterations Count: 7


### Step 6 — Compute Model BIC Score

In [64]:
bic_score_61 = gmm_61.bic(X_57)

In [65]:
print('[Day 061] Model BIC Score:', bic_score_61)

[Day 061] Model BIC Score: 5412.8521
